# 02 Comparing groups: t-tests, analysis of variance, ranks and spreads

simplydata's words for comparing groups, checked on
- **Student's sleep data** (Cushny & Peebles 1905, analysed by Student 1908): the published averages;
- the eleven **NIST StRD one-way ANOVA** datasets, whose F statistic and degrees of freedom are certified;
- **iris** (Anderson 1935, Fisher 1936) and **warpbreaks** (Tippett 1950), for Welch's ANOVA, the rank test and the
  tests of equal spread.

**Only simplydata runs in this notebook.** The reference values are in `references/02_comparing_groups.csv`, written
by `scripts/references/02_comparing_groups.py`: the printed and certified values first (published numbers allowing
for their printed rounding; NIST to at least 9 significant digits, LRE); only where nothing is printed, R run once
there (`t.test`, `anova(lm())`, `oneway.test`, `kruskal.test`, `bartlett.test`, `fligner.test`, and Levene's test as
an ANOVA of the distances from the group medians), to a relative difference of 1e-10. Each check prints its source.

In [1]:
import simplydata as sdata

from checks import ReferenceChecks, data_file, quiet, read_row, versions

NOTEBOOK = "02_comparing_groups"
versions("simplydata")
checks = ReferenceChecks(NOTEBOOK)


def load(name):
    return quiet(sdata.table, data_file(NOTEBOOK, name), separator=",")


def check_row(dataset, method, result, names):
    """Compare the named columns of the first row of a simplydata result (read with table.get_row)."""
    row = read_row(result)
    checks.check_values(dataset, method, {name: row[column] for name, column in names.items()})

Python 3.12.16 on Linux 7.0.0-1020-azure
simplydata 0.2.0 (commit a5a8ff0 of github.com/bioforkgithub/simplydata)
simplydata 0.2.0


R version 4.6.2 (2026-10-29)


## Student's sleep data

Extra hours of sleep in 10 patients, each given two drugs. Student (1908) printed the averages +0.75 and +2.33
hours and the average difference within patients, +1.58 hours.

In [2]:
sleep = load("sleep")
sleep.show_first_rows()

,extra,group,ID
1,0.7,drug 1,1
2,-1.6,drug 1,2
3,-0.2,drug 1,3
4,-1.2,drug 1,4
5,-0.1,drug 1,5


### Two separate groups: `compare_averages_of` (Welch's t-test)

The same underneath: SciPy `ttest_ind(equal_var=False)`. (The data are paired, so this is not the right analysis of
them; it checks the arithmetic of the two-group test. simplydata's difference is drug 1 minus drug 2.)

In [3]:
result = sleep.compare_averages_of("extra", for_each_group_in="group")

The average "extra" is 0.750 in drug 1 and 2.33 in drug 2: drug 1 is 1.58 lower (95 % CI of the difference -3.37 to 0.205; Hedges' g = -0.797 (95 % CI -1.67 to 0.0914), a difference of 0.797 standard deviations).
If there were truly no difference, a difference this big or bigger would turn up in about 8 of 100 studies like this one (p = 0.079). This does not show that there is no difference.
Test: Welch's t-test (t = -1.86, df = 17.8), the usual test for two averages; it does not assume both groups have the same spread.


In [4]:
QUANTITIES = ["average in drug 1", "average in drug 2", "difference", "statistic", "degrees of freedom", "p-value"]
check_row("Student's sleep data", "compare_averages_of (Welch)", result, {q: q for q in QUANTITIES})

agrees  Student's sleep data | compare_averages_of (Welch) | average in drug 1: simplydata 0.75, Student (1908), 2 decimals, printed 0.75; difference 0.00e+00 (allowed 5e-03)
      Source: Student (1908), 2 decimals, printed (printed) - https://doi.org/10.1093/biomet/6.1.1
agrees  Student's sleep data | compare_averages_of (Welch) | average in drug 2: simplydata 2.33, Student (1908), 2 decimals, printed 2.33; difference 4.44e-16 (allowed 5e-03)
      Source: Student (1908), 2 decimals, printed (printed) - https://doi.org/10.1093/biomet/6.1.1
agrees  Student's sleep data | compare_averages_of (Welch) | difference: simplydata -1.58, Student (1908), 2 decimals, printed -1.58; difference 4.44e-16 (allowed 5e-03)
      Source: Student (1908), 2 decimals, printed (printed) - https://doi.org/10.1093/biomet/6.1.1
agrees  Student's sleep data | compare_averages_of (Welch) | statistic: simplydata -1.86081346749, R t.test -1.86081346749; relative difference 1.19e-16 (allowed 1e-10)
      Source: 

### The same patients twice: `compare_before_and_after` (paired t-test)

The analysis Student made. The same underneath: SciPy `ttest_rel`.

In [5]:
result = load("sleep_wide").compare_before_and_after("drug 1", "drug 2")

On average "drug 2" is 1.58 higher than "drug 1" (95 % CI of the average change 0.700 to 2.46; Cohen's d_z = 1.28 (95 % CI 0.415 to 2.12)).
If there were truly no change, a change this big or bigger would turn up in about 3 of 1000 studies like this one (p = 0.003).
Test: paired t-test (t = 4.06, df = 9), the usual test when each row is measured twice; it looks at the change in each row.
Note: the changes don't look bell-shaped (Shapiro-Wilk p = 0.033); a safer test here: based_on="ranks".


In [6]:
check_row("Student's sleep data", "compare_before_and_after", result,
          {q: q for q in ["average change", "statistic", "degrees of freedom", "p-value"]})

agrees  Student's sleep data | compare_before_and_after | average change: simplydata 1.58, Student (1908), 2 decimals, printed 1.58; difference 0.00e+00 (allowed 5e-03)
      Source: Student (1908), 2 decimals, printed (printed) - https://doi.org/10.1093/biomet/6.1.1
agrees  Student's sleep data | compare_before_and_after | statistic: simplydata 4.06212768338, R t.test(paired = TRUE) 4.06212768338; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test(paired = TRUE) (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  Student's sleep data | compare_before_and_after | degrees of freedom: simplydata 9, R t.test(paired = TRUE) 9; relative difference 0.00e+00 (allowed 1e-10)
      Source: R t.test(paired = TRUE) (run reference) - https://search.r-project.org/R/refmans/stats/html/t.test.html
agrees  Student's sleep data | compare_before_and_after | p-value: simplydata 0.00283289019738, R t.test(paired = TRUE) 0.00283289019738; relative differ

## NIST one-way analysis of variance: `compare_averages_of(test="anova")`

The classic one-way ANOVA (equal spreads assumed). NIST certifies the F statistic and its degrees of freedom; the
p-value is compared with R `anova(lm(y ~ group))`. SmLs01 to SmLs09 grow harder: the same groups shifted to ever
larger numbers (1e6 and 1e12 added), with more rows. NIST rates SiRstv and SmLs01-03 lower difficulty, AtmWtAg and
SmLs04-06 average, SmLs07-09 higher.

In [7]:
NIST_ANOVA = ["SiRstv", "SmLs01", "SmLs02", "SmLs03", "AtmWtAg", "SmLs04", "SmLs05", "SmLs06",
              "SmLs07", "SmLs08", "SmLs09"]
anova_data = {name: load(f"nist_{name}") for name in NIST_ANOVA}
anova_data["AtmWtAg"].compare_averages_of("y", for_each_group_in="group", test="anova")

Note: "test=" is now called "test_method=". The old name will stop working in a later version.
The average "y" is 108 in 1 and 108 in 2. Omega-squared = 0.237 (95 % CI 0.0566 to 0.429): the groups explain about 24 % of the differences in "y".
If the groups were truly alike, differences this big or bigger between them would turn up in fewer than 1 in 1000 studies like this one (p = 0.0002).
Test: one-way ANOVA (F = 15.9, df = 1 and 46), chosen with test_method=; it assumes the groups have the same spread.


,test,what it compares,statistic name,statistic,degrees of freedom 1,degrees of freedom 2,p-value,effect size name,effect size,effect size 95 % CI low,effect size 95 % CI high,count in 1,average in 1,count in 2,average in 2
1,one-way ANOVA,"the average of ""y"" in each ""group"" (1, 2)",F,15.946734,1.0,46.0,0.000233,share of the differences explained by the grou...,0.237451,0.056556,0.428521,24,107.868154,24,107.868136


In [8]:
for name, table in anova_data.items():
    result = quiet(table.compare_averages_of, "y", for_each_group_in="group", test="anova")
    check_row(f"NIST {name}", "compare_averages_of (ANOVA)", result,
              {"F": "statistic", "between df": "degrees of freedom 1", "within df": "degrees of freedom 2",
               "p-value": "p-value"})

agrees  NIST SiRstv | compare_averages_of (ANOVA) | F: simplydata 1.1804623744, NIST certified 1.1804623744; LRE 13.1 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SiRstv.html
agrees  NIST SiRstv | compare_averages_of (ANOVA) | between df: simplydata 4, NIST certified 4; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SiRstv.html
agrees  NIST SiRstv | compare_averages_of (ANOVA) | within df: simplydata 20, NIST certified 20; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SiRstv.html
agrees  NIST SiRstv | compare_averages_of (ANOVA) | p-value: simplydata 0.349447493402, R anova(lm()) 0.349447493402; relative difference 1.87e-13 (allowed 1e-10)
      Source: R anova(lm()) (run reference) - https://search.r-project.org/R/refmans/stats/html/anova.html
agrees  NIST SmLs01 | compare_averages_o

agrees  NIST SmLs03 | compare_averages_of (ANOVA) | F: simplydata 2001, NIST certified 2001; LRE 13.6 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs03.html
agrees  NIST SmLs03 | compare_averages_of (ANOVA) | between df: simplydata 8, NIST certified 8; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs03.html
agrees  NIST SmLs03 | compare_averages_of (ANOVA) | within df: simplydata 18000, NIST certified 18000; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs03.html
agrees  NIST SmLs03 | compare_averages_of (ANOVA) | p-value: simplydata 0, R anova(lm()) 0; relative difference 0.00e+00 (allowed 1e-10)
      Source: R anova(lm()) (run reference) - https://search.r-project.org/R/refmans/stats/html/anova.html


agrees  NIST AtmWtAg | compare_averages_of (ANOVA) | F: simplydata 15.9467335667, NIST certified 15.9467335678; LRE 10.2 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/AtmWtAg.html
agrees  NIST AtmWtAg | compare_averages_of (ANOVA) | between df: simplydata 1, NIST certified 1; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/AtmWtAg.html
agrees  NIST AtmWtAg | compare_averages_of (ANOVA) | within df: simplydata 46, NIST certified 46; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/AtmWtAg.html
explained difference – reference (C2)  NIST AtmWtAg | compare_averages_of (ANOVA) | p-value: simplydata 0.000232684448441, R anova(lm()) 0.000232684448067; relative difference 1.61e-09 (allowed 1e-10)
      why: NIST AtmWtAg: F and p-value against R anova(lm()) - https://github.com/bioforkgithub/simpl

agrees  NIST SmLs05 | compare_averages_of (ANOVA) | F: simplydata 201.000000012, NIST certified 201; LRE 10.2 (needs >= 9)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs05.html
agrees  NIST SmLs05 | compare_averages_of (ANOVA) | between df: simplydata 8, NIST certified 8; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs05.html
agrees  NIST SmLs05 | compare_averages_of (ANOVA) | within df: simplydata 1800, NIST certified 1800; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs05.html
agrees  NIST SmLs05 | compare_averages_of (ANOVA) | p-value: simplydata 4.03714178023e-243, R anova(lm()) 4.03714178023e-243; relative difference 6.51e-13 (allowed 1e-10)
      Source: R anova(lm()) (run reference) - https://search.r-project.org/R/refmans/stats/html/anova.html
agrees  NIST SmLs06 | compare_averag

explained difference – precision limit (B2)  NIST SmLs08 | compare_averages_of (ANOVA) | F: simplydata 201.013004096, NIST certified 201; LRE 4.2 (needs >= 9)
      why: One-way ANOVA on NIST SmLs07, SmLs08, SmLs09: F agrees to 4.2 to 4.4 digits - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#b2-one-way-anova-on-nist-smls07-smls08-smls09-f-agrees-to-42-to-44-digits
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs08.html
agrees  NIST SmLs08 | compare_averages_of (ANOVA) | between df: simplydata 8, NIST certified 8; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs08.html
agrees  NIST SmLs08 | compare_averages_of (ANOVA) | within df: simplydata 1800, NIST certified 1800; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs08.html
explained difference – precision limit (B2)

explained difference – precision limit (B2)  NIST SmLs09 | compare_averages_of (ANOVA) | F: simplydata 2001.13492622, NIST certified 2001; LRE 4.2 (needs >= 9)
      why: One-way ANOVA on NIST SmLs07, SmLs08, SmLs09: F agrees to 4.2 to 4.4 digits - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#b2-one-way-anova-on-nist-smls07-smls08-smls09-f-agrees-to-42-to-44-digits
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs09.html
agrees  NIST SmLs09 | compare_averages_of (ANOVA) | between df: simplydata 8, NIST certified 8; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs09.html
agrees  NIST SmLs09 | compare_averages_of (ANOVA) | within df: simplydata 18000, NIST certified 18000; difference 0.00e+00 (allowed 0e+00)
      Source: NIST certified (certified) - https://www.itl.nist.gov/div898/strd/anova/SmLs09.html
agrees  NIST SmLs09 | compare_averages_o

## Welch's ANOVA (the default for three or more groups): iris

Sepal length of the three iris species. `compare_averages_of` without `test=` uses Welch's one-way ANOVA, which does
not assume equal spreads. The same underneath: statsmodels `anova_oneway(use_var="unequal")`; compared with R
`oneway.test()`.

In [9]:
iris = quiet(sdata.table, "../data/iris.csv")
result = iris.compare_averages_of("Sepal.Length", for_each_group_in="Species")

The average "Sepal.Length" is 5.01 in setosa, 5.94 in versicolor and 6.59 in virginica. Omega-squared = 0.612 (95 % CI 0.517 to 0.683): the groups explain about 61 % of the differences in "Sepal.Length".
If the groups were truly alike, differences this big or bigger between them would turn up in fewer than 1 in 1000 studies like this one (p < 0.0001).
Test: Welch's one-way ANOVA (F = 139, df = 2 and 92.2), the usual test for three or more averages; it does not assume the groups have the same spread. To see which groups differ: table.compare_each_pair_of("Sepal.Length", for_each_group_in="Species").


In [10]:
check_row("iris", "compare_averages_of (Welch ANOVA)", result,
          {q: q for q in ["statistic", "degrees of freedom 1", "degrees of freedom 2", "p-value"]})

agrees  iris | compare_averages_of (Welch ANOVA) | statistic: simplydata 138.908285269, R oneway.test 138.908285269; relative difference 1.84e-15 (allowed 1e-10)
      Source: R oneway.test (run reference) - https://search.r-project.org/R/refmans/stats/html/oneway.test.html
agrees  iris | compare_averages_of (Welch ANOVA) | degrees of freedom 1: simplydata 2, R oneway.test 2; relative difference 0.00e+00 (allowed 1e-10)
      Source: R oneway.test (run reference) - https://search.r-project.org/R/refmans/stats/html/oneway.test.html
agrees  iris | compare_averages_of (Welch ANOVA) | degrees of freedom 2: simplydata 92.2111453205, R oneway.test 92.2111453205; relative difference 3.08e-16 (allowed 1e-10)
      Source: R oneway.test (run reference) - https://search.r-project.org/R/refmans/stats/html/oneway.test.html
agrees  iris | compare_averages_of (Welch ANOVA) | p-value: simplydata 1.50505896275e-28, R oneway.test 1.50505896275e-28; relative difference 5.29e-14 (allowed 1e-10)
      Sou

## Ranks: `compare_averages_of(based_on="ranks")` (Kruskal-Wallis)

Warp breaks per loom at three tensions. The same underneath: SciPy `kruskal`; compared with R `kruskal.test()`.

In [11]:
warp = quiet(sdata.table, "../data/warpbreaks.csv")
result = warp.compare_averages_of("breaks", for_each_group_in="tension", based_on="ranks")

The median "breaks" is 20.5 in H, 29.5 in L and 27.0 in M. Epsilon-squared = 0.204 (95 % bootstrap CI, R seed 1, 0.0576 to 0.421): the groups explain about 20 % of the differences in the ranks.
If the groups were truly alike, differences this big or bigger between them would turn up in about 4 of 1000 studies like this one (p = 0.004).
Test: Kruskal-Wallis test (H = 10.8, df = 2), which compares ranks rather than averages, so it does not need bell-shaped values. To see which groups differ: table.compare_each_pair_of("breaks", for_each_group_in="tension", based_on="ranks").


In [12]:
check_row("warpbreaks", "compare_averages_of (Kruskal-Wallis)", result, {"statistic": "statistic", "p-value": "p-value"})

agrees  warpbreaks | compare_averages_of (Kruskal-Wallis) | statistic: simplydata 10.8092652706, R kruskal.test 10.8092652706; relative difference 0.00e+00 (allowed 1e-10)
      Source: R kruskal.test (run reference) - https://search.r-project.org/R/refmans/stats/html/kruskal.test.html
agrees  warpbreaks | compare_averages_of (Kruskal-Wallis) | p-value: simplydata 0.00449570566138, R kruskal.test 0.00449570566138; relative difference 1.23e-14 (allowed 1e-10)
      Source: R kruskal.test (run reference) - https://search.r-project.org/R/refmans/stats/html/kruskal.test.html


## Spreads: `compare_spreads_of`

Do the three tensions differ in spread? Levene's test centred on the median (the default; SciPy
`levene(center="median")` underneath), compared with the same test written out in base R (an ANOVA of the distances
from each group's median, as `car::leveneTest()`); Bartlett's test against R `bartlett.test()`; Fligner-Killeen
against R `fligner.test()`.

In [13]:
for test in ("levene", "bartlett", "fligner"):
    result = quiet(warp.compare_spreads_of, "breaks", for_each_group_in="tension", test=test)
    check_row("warpbreaks", f"compare_spreads_of ({test})", result, {"statistic": "statistic", "p-value": "p-value"})

agrees  warpbreaks | compare_spreads_of (levene) | statistic: simplydata 2.81804766589, R anova of distances from medians 2.81804766589; relative difference 6.30e-16 (allowed 1e-10)
      Source: R anova of distances from medians (run reference) - https://search.r-project.org/R/refmans/stats/html/anova.html
agrees  warpbreaks | compare_spreads_of (levene) | p-value: simplydata 0.0690494519399, R anova of distances from medians 0.0690494519399; relative difference 1.00e-15 (allowed 1e-10)
      Source: R anova of distances from medians (run reference) - https://search.r-project.org/R/refmans/stats/html/anova.html
agrees  warpbreaks | compare_spreads_of (bartlett) | statistic: simplydata 9.65711813841, R bartlett.test 9.65711813841; relative difference 2.94e-15 (allowed 1e-10)
      Source: R bartlett.test (run reference) - https://search.r-project.org/R/refmans/stats/html/bartlett.test.html
agrees  warpbreaks | compare_spreads_of (bartlett) | p-value: simplydata 0.0079980375855, R bartl

## Result

In [14]:
checks.finish()

02_comparing_groups: 66 comparisons - 60 agrees, 1 explained difference – reference, 5 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,02_comparing_groups,Student's sleep data,compare_averages_of (Welch),average in drug 1,0.750000,0.750000,"Student (1908), 2 decimals, printed",0.000000e+00,5.000000e-03,absolute,0.000000e+00,PASS,,agrees,,,,printed,https://doi.org/10.1093/biomet/6.1.1,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
1,02_comparing_groups,Student's sleep data,compare_averages_of (Welch),average in drug 2,2.330000,2.330000,"Student (1908), 2 decimals, printed",-4.440892e-16,5.000000e-03,absolute,4.440892e-16,PASS,,agrees,,,,printed,https://doi.org/10.1093/biomet/6.1.1,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
2,02_comparing_groups,Student's sleep data,compare_averages_of (Welch),difference,-1.580000,-1.580000,"Student (1908), 2 decimals, printed",4.440892e-16,5.000000e-03,absolute,4.440892e-16,PASS,,agrees,,,,printed,https://doi.org/10.1093/biomet/6.1.1,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
3,02_comparing_groups,Student's sleep data,compare_averages_of (Welch),statistic,-1.860813,-1.860813,R t.test,2.220446e-16,1.000000e-10,relative,1.193266e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
4,02_comparing_groups,Student's sleep data,compare_averages_of (Welch),degrees of freedom,17.776474,17.776474,R t.test,-1.065814e-14,1.000000e-10,relative,5.995644e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,02_comparing_groups,warpbreaks,compare_spreads_of (levene),p-value,0.069049,0.069049,R anova of distances from medians,6.938894e-17,1.000000e-10,relative,1.004917e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
62,02_comparing_groups,warpbreaks,compare_spreads_of (bartlett),statistic,9.657118,9.657118,R bartlett.test,2.842171e-14,1.000000e-10,relative,2.943084e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
63,02_comparing_groups,warpbreaks,compare_spreads_of (bartlett),p-value,0.007998,0.007998,R bartlett.test,-2.602085e-17,1.000000e-10,relative,3.253405e-15,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
64,02_comparing_groups,warpbreaks,compare_spreads_of (fligner),statistic,3.901542,3.901542,R fligner.test,0.000000e+00,1.000000e-10,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/R/refmans/stats/h...,"GitHub Actions, Linux 7.0.0-1020-azure, Intel(..."
